# 01 - Download LibriSpeech

Pulls the LibriSpeech audio subsets into the Colab VM's **local scratch** (`/content/data`, fast and ~100 GB, but wiped when the session ends). Downloads are resumable across dropped sessions and md5-verified against OpenSLR's official checksums.

This notebook is a thin launcher: the real code lives in `training/data/librispeech.py` in the repo.

## 1. Pull the repo into the VM

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/AndrewD0/enph-479-stt-neural-network.git"
REPO_DIR = "/content/enph-479-stt-neural-network"

# Public repo, so no auth needed.
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

sys.path.insert(0, REPO_DIR)
print("repo ready:", REPO_DIR)

## 2. Download the audio into `/content/data`

`train-clean-100` is the acoustic-model training set to start with; `dev-clean` and `test-clean` are for decode-weight tuning and final eval. To scale up later (if the WER target needs it), add `train-clean-360` and/or `train-other-500` to the list. The language-model text corpus is a separate pull (see the last cell); it is not needed to train the acoustic model.

In [ ]:
from training.data.librispeech import download

SUBSETS = ["train-clean-100", "dev-clean", "test-clean"]
DATA_DIR = "/content/data"  # local scratch, NOT Drive (Drive random reads starve the GPU)

paths = download(SUBSETS, dest=DATA_DIR)
paths

## 3. Verify what landed

In [ ]:
%%bash
echo "== sizes =="
du -sh /content/data/LibriSpeech/* 2>/dev/null
echo
echo "== flac count per subset =="
for d in /content/data/LibriSpeech/*/; do printf "%s: " "$(basename "$d")"; find "$d" -name '*.flac' | wc -l; done

## 4. (Later) Language-model text corpus

Needed for the char-LM and KenLM word-LM stages, not for acoustic-model training. Uncomment when you get there. The pruned ARPA (13 MB) is enough for a first end-to-end decode; the full normalized text (1.5 GB) is only needed to build a custom LM.

In [ ]:
# from training.data.librispeech import download_lm
# download_lm(["3-gram.pruned.3e-7.arpa.gz", "librispeech-vocab.txt"], dest="/content/data/lm")